In [75]:
import pandas as pd
import yaml
import os
from pathlib import Path
from pprint import pprint

In [85]:
file = '/home/ec2-user/SageMaker/auto-train/dealer quotation/Invoice Data- Annotations Final.xlsx'
folder = "/home/ec2-user/SageMaker/auto-train/dealer quotation/Bulk Document List_Performa invoice"
datadf = pd.read_excel(file)

In [86]:
datadf['file_path'] = datadf['doc_id'] \
    .apply(lambda x:  Path(folder) / x / (os.listdir(Path(folder) / x)[0]))

In [90]:
datadf.dealer_stamp.value_counts()

dealer_stamp
Availability    50
Name: count, dtype: int64

In [88]:
datadf.to_csv("/home/ec2-user/SageMaker/auto-train/dealer quotation/dealer_quotation_train_data.csv", index=False)

In [56]:
config = {}
with open("protein/config.yaml") as file:
    config = yaml.safe_load(file)

In [57]:
def flatten_and_calibrate_prompt(obj, prefix):
    lines = []

    def _flatten(x, prefix):
        if isinstance(x, dict):
            for k, v in x.items():
                _flatten(v, f"{prefix}{k}: ")
        elif isinstance(x, list):
            for item in x:
                _flatten(item, prefix)
        else:
            lines.append(f"{prefix}{x}")

    _flatten(obj, prefix)
    return " ".join(lines)


In [67]:
import json

def flatten_grouped_with_schema(d):
    lines = []

    def walk(key, value, indent=0):
        prefix = "  " * indent

        # 🔹 Special handling for Schema
        if key == "Schema" and isinstance(value, dict):
            lines.append(f"{prefix}{key}:")
            schema_str = json.dumps(value, indent=2)
            for line in schema_str.splitlines():
                lines.append(f"{prefix}  {line}")
            return

        if isinstance(value, dict):
            lines.append(f"{prefix}{key}:")
            for k, v in value.items():
                walk(k, v, indent + 1)

        elif isinstance(value, list):
            lines.append(f"{prefix}{key}:")
            for item in value:
                if isinstance(item, dict):
                    for k, v in item.items():
                        walk(k, v, indent + 1)
                else:
                    lines.append(f"{prefix}  - {item}")

        else:
            lines.append(f"{prefix}{key}: {value}")

    for k, v in d.items():
        print(k)
        if k not in ['prompt_tokens', 'completion_tokens', 'total_tokens', 'cost']:
            walk(k, v)

    return "\n".join(lines)


In [68]:
a = {'Task': 'Extract PAN card entities using OCR text and bbox with spatial reasoning.', 'Goal': 'Return schema-aligned PAN fields via label anchoring and layout validation.', 'Schema': {'Name': 'string|null', 'PAN': 'string|null', 'DOB': 'string|null', 'Signature': 'Present|Absent|Not Applicable'}, 'Guardrails': ["Document-type confirmation: Near the top header, detect any of 'INCOME TAX DEPARTMENT', 'GOVT. OF INDIA', 'Permanent Account Number'/'PERMANENT ACCOUNT NUMBER'/'Permanent Account Number Card'. If none found, return null for all fields.", 'Coordinate normalization: Verify bboxes are normalized to image width/height; y increases downward. Validate scale using the PAN text bbox height as a reference. Deskew using card borders before region proposals.', 'Orientation via hologram/QR/photo: QR/hologram typically sits in the right half; treat the column left-of-QR/hologram as the personal-details region for Name, DOB/DOI, PAN.', 'Label anchoring: Prefer values on the same baseline or the immediate line right/below their label with vertical gap ≤ 1 line-height or ≤ 5% of card height.', 'Line clustering: Form single-line bands using a vertical-gap threshold (< one line-height or < 5% of card height). Do not merge stacked lines.', 'Left-margin consistency: For Name, DOB/DOI, and PAN, prefer candidates sharing a common left margin inside the personal-details column; downweight tokens overlapping the QR/hologram cluster.', 'Token integrity: For PAN/date, never merge across separate lines. Allow PAN concatenation only for adjacent tokens on the same baseline that exactly match the PAN regex.', 'Single-source rule: Extract Name, PAN, and Date from the left personal-details cluster relative to QR/hologram. Only extract elsewhere if explicitly anchored by a matching label bbox.', "Entity-type consistency: Use PAN 4th character type code ('P','C','H','A','B','G','J','L','F','T') to select DOB (individual) vs DOI (non-individual) and Signature applicability.", "Signature ROI and stroke detection: After DOB, expand search toward the lower-right quadrant beneath DOB and near the QR/seal area; ensure the card’s bottom edge is included. Anchor to a 'Signature' label or horizontal line artifact. Apply adaptive thresholding and morphological smoothing to detect compact, continuous pen-stroke clusters; exclude periodic hologram textures and printed text. Deskew first.", 'Confidence and no hallucination: Prefer OCR confidence > 0.6. Emit null/Not Applicable if evidence is not explicit and spatially aligned. Do not trim or normalize Name spacing.'], 'Entities to be extracted': {'Name': {'key_definitions': 'Primary holder/entity name in the personal-details column, typically above DOB/DOI and PAN.', 'reasoning': ['Step 1: Confirm PAN card via header labels; orient using QR/hologram bboxes. Define the personal-details column as the area left-of-QR/hologram.', "Step 2: If a 'Name' label exists, set candidate region to the same baseline or the immediate line right/below the label with a small vertical gap. Select the longest contiguous uppercase span on that line; include initials only if on the same baseline.", "Step 3: If no 'Name' label, find the top-most single line directly below the header within the personal-details column. Prefer bold/uppercase text above the date and PAN labels and aligned to the column’s left margin.", 'Step 4: Enforce single-line integrity: merge only tokens sharing near-equal y (same baseline band). Do not merge stacked lines; if two stacked lines exist, select the upper one.', "Step 5: Exclude guardian/company adjunct lines: if 'Father's Name'/'S/W/D' labels are nearby below, restrict Name to the line immediately above those labels. Remove titles (MR/MRS/MS) if separated by a clear token boundary on the same line.", "Step 6: Cross-validate with PAN type (4th char). If not 'P', prefer organization lines containing PRIVATE, LIMITED, LTD, LLP, HUF, TRUST, CO-OP, COMPANY on the same baseline. Do not concatenate across lines."]}, 'PAN': {'key_definitions': "Ten-character alphanumeric identifier adjacent to the 'Permanent Account Number' label.", 'reasoning': ["Step 1: Locate 'Permanent Account Number'/'PERMANENT ACCOUNT NUMBER'/'Permanent Account Number Card' label bbox. Define PAN region as same line or immediate line right/below the label with minimal vertical gap.", 'Step 2: Within/adjacent to this region, select the token matching regex [A-Z]{5}[0-9]{4}[A-Z}. Allow concatenation only if adjacent tokens on the same baseline combine to exactly match the regex.', 'Step 3: Apply proximity: choose the candidate with highest IoU or closest centroid to the PAN label bbox, constrained to the personal-details column and card-core hull. Reject tokens overlapping the QR/hologram or distant clusters without direct label anchoring.', "Step 4: Validate: ensure length = 10; letters A–Z; digits 0–9; confirm 4th character ∈ {P,C,H,A,B,G,J,L,F,T}. Avoid 'O'↔'0' substitutions.", 'Step 5: If label missing, search the personal-details cluster (below Name and Date) for a single regex-valid token aligned to the column’s left margin. Reject header/marginal matches outside the card-core hull.', 'Step 6: Prefer higher OCR confidence (>0.6); normalize to uppercase. If no valid PAN after checks, return null.']}, 'DOB': {'key_definitions': "For individuals (PAN 4th char 'P'): Date of Birth; otherwise DOI/Formation. Printed as DD/MM/YYYY.", 'reasoning': ["Step 1: Determine PAN entity type via 4th character: 'P' → individual (DOB), others → non-individual (DOI/Formation).", "Step 2: Search for date labels: 'Date of Birth'/'DOB'/'DATE OF BIRTH' and 'Date of Incorporation'/'Incorporation/Formation'. Anchor the value on the same line or immediate line right/below the detected label bbox, aligned to the column’s left margin.", "Step 3: Identify candidate tokens matching DD/MM/YYYY. If separators are '-', '.', or spaces but digits map to DD MM YYYY, normalize to '/'. Validate day (01–31), month (01–12), year (1900–2099). Reject incomplete or multi-line dates.", 'Step 4: Use vertical order: select the date positioned below the Name and above the PAN label, closest to its respective date label; de-weight dates near right-side stamps, vertical text, or QR/hologram.', 'Step 5: If both DOB and DOI labels exist, choose the one consistent with PAN type and nearer to the name block. If labels are absent, pick the nearest valid date in the left personal-details cluster above the PAN label within the card-core hull.', 'Step 6: Final validation: ensure the chosen date’s bbox aligns to the personal-details column left margin. If multiple dates match, select the one with the highest label proximity and OCR confidence.']}, 'Signature': {'key_definitions': "Presence of handwritten signature near the 'Signature' label or typical lower panel; applicable only for individual PAN ('P').", 'reasoning': ["Step 1: Applicability: If PAN 4th char ≠ 'P', set 'Not Applicable' and stop.", 'Step 2: Normalize and deskew: Confirm bbox-to-image normalization using the PAN text bbox as a scale reference. Deskew using card borders so expected regions remain within bounds.', "Step 3: Anchor search: From the 'INCOME TAX DEPARTMENT' header bbox, validate orientation. After extracting DOB, sweep downward and expand ROI toward the lower-right quadrant beneath DOB and near the QR/seal area. Ensure the card’s bottom edge is fully included to avoid cropping the signature line.", "Step 4: Target artifacts: Prioritize areas near a 'Signature' label or a faint horizontal line artifact typical of signature panels. Center the ROI on the label bbox and the immediate lower area when present.", 'Step 5: Ink-stroke detection: Apply adaptive thresholding and morphological smoothing to detect compact, continuous cursive pen-stroke clusters over patterned backgrounds. Prefer connected components with irregular contours and varying width.', 'Step 6: Exclusions: Exclude periodic hologram textures, photo edges, printed text, stamps, and vertical text. Reject clusters outside the signature ROI or far right without linkage to the label/line.', "Step 7: Evidence check: Set 'Present' only if stroke components overlap the ROI or lie immediately below the 'Signature' label/line in the lower panel. If no strokes are found, set 'Absent'."]}}, 'Foot Notes': ['Template cues: Header near top; QR/hologram typically in the right half; photo may be above/beside the signature panel depending on variant.', 'Baseline estimation: Use median token height to approximate line-height for vertical-gap thresholds.', 'Card-core hull: Convex hull of tokens excluding outlier stamps and hologram areas helps prevent misaligned extractions.', 'Iterative validation: Cross-check each extracted value’s bbox proximity to its label, column alignment, and expected vertical order before finalizing.'], 'Failure Cases': [{'Input': 'SIGNATURE label bbox present; no pen strokes below label; hologram periodic pattern overlaps ROI.', 'Output': "Signature='Absent'"}, {'Input': "Left-column 'DOB' label with '05/07/1988'; right-side stamp shows '12/10/2019'.", 'Output': "DOB='05/07/1988'"}, {'Input': "Adjacent tokens 'ABCDE1' and '234F' on same baseline under PAN label.", 'Output': "PAN='ABCDE1234F'"}, {'Input': "Two stacked uppercase lines under header: 'RAHUL KUMAR' (upper) and 'VIJAY KUMAR' (lower); guardian label below.", 'Output': "Name='RAHUL KUMAR'"}, {'Input': "Non-individual PAN ('C' as 4th char); 'Signature' label present; no strokes.", 'Output': "Signature='Not Applicable'"}, {'Input': 'Slight rotation pushes expected signature line outside initial ROI; deskew applied.', 'Output': "Signature='Present' after deskew and ROI expansion to lower-right beneath DOB."}], 'prompt_tokens': 3549, 'completion_tokens': 4415, 'total_tokens': 7964, 'cost': 3.8869}

In [70]:
print(flatten_grouped_with_schema(a))

Task
Goal
Schema
Guardrails
Entities to be extracted
Foot Notes
Failure Cases
prompt_tokens
completion_tokens
total_tokens
cost
Task: Extract PAN card entities using OCR text and bbox with spatial reasoning.
Goal: Return schema-aligned PAN fields via label anchoring and layout validation.
Schema:
  {
    "Name": "string|null",
    "PAN": "string|null",
    "DOB": "string|null",
    "Signature": "Present|Absent|Not Applicable"
  }
Guardrails:
  - Document-type confirmation: Near the top header, detect any of 'INCOME TAX DEPARTMENT', 'GOVT. OF INDIA', 'Permanent Account Number'/'PERMANENT ACCOUNT NUMBER'/'Permanent Account Number Card'. If none found, return null for all fields.
  - Coordinate normalization: Verify bboxes are normalized to image width/height; y increases downward. Validate scale using the PAN text bbox height as a reference. Deskew using card borders before region proposals.
  - Orientation via hologram/QR/photo: QR/hologram typically sits in the right half; treat the co

In [33]:
data = pd.read_csv("/home/ec2-user/SageMaker/auto-train/prompt_tuner/gt_protein.csv")

In [34]:
data['file_path'] = data['file_path'].apply(lambda x: os.path.join("/home/ec2-user/SageMaker/auto-train/prompt_tuner/", x))

In [35]:
data.to_csv("/home/ec2-user/SageMaker/auto-train/prompt_tuner/CUW/gt_protein.csv", index=False)

In [36]:
data

,Signature,file_path,DOB,Name,PAN
0,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,NaN,NaN,NaN
1,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,01/01/1994,SANDIP KUMAR,LQBPK3185J
2,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,07/06/2003,PHOOL KUMAR,LLVPK8896H
3,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,16/11/1989,KARTHIK,EGSPK8900D
4,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,11/11/1988,AZMAN,EHWPA3405G
...,...,...,...,...,...
170,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,15/06/2001,BUDHARAPU DEEPAK,EYPPB3189A
171,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,16/06/2004,VANSH,CKQPV5928L
172,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,14/10/2000,Vignesh,CZCPV7087D
173,Present,/home/ec2-user/SageMaker/auto-train/prompt_tun...,01/01/1988,Sunitha,RHJPS6991D
